# RAG Pipeline

## Install Dependencies

In [1]:
!pip install -q transformers sentence-transformers faiss-cpu wikipedia pandas numpy tqdm datasets scikit-learn


[notice] A new release of pip is available: 25.3 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


## Imports

In [2]:
import json
import re
import numpy as np
import pandas as pd
from tqdm import tqdm
import wikipedia
import faiss

from sentence_transformers import SentenceTransformer, CrossEncoder
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM


/Library/Frameworks/Python.framework/Versions/3.14/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Configuration

These hyperparameters define the behavior of the RAG pipeline, including chunk granularity, retrieval depth, embedding generation, reranking configuration, and language model selection. The values were tuned to balance retrieval quality, answer grounding, computational efficiency, and memory limitations.

In [3]:
MAX_SAMPLES = 100 # how many evaluation samples to process

CHUNK_SIZE = 100 # how many words each chunk contains
OVERLAP = 30 # how many words overlap between adjacent chunks

TOP_K = 20 # how many chunks FAISS retrieves initially
TOP_N = 5 # final chunks used after reranking

EMBEDDING_MODEL = "BAAI/bge-small-en-v1.5" # which embedding model generates vectors : text → vector
GENERATION_MODEL = "google/flan-t5-base" # which LLM generates answers


## Load Dataset

This block loads the Natural Questions dataset from JSONL format, parses each question-answer pair, limits the number of evaluation samples for computational efficiency, and converts the data into a Pandas DataFrame for downstream retrieval, generation, and evaluation.

In [4]:
DATA_PATH = "NQ-open.dev.jsonl"

def load_jsonl(path, max_samples=None): # a custom function for loading JSONL files

    data = []

    with open(path, 'r', encoding='utf-8') as f:

        for idx, line in enumerate(f):

            if max_samples and idx >= max_samples:
                break

            data.append(json.loads(line))

    return pd.DataFrame(data)

df = load_jsonl(DATA_PATH, MAX_SAMPLES)

df.head()


,question,answer
0,when was the last time anyone was on the moon,"[14 December 1972 UTC, December 1972]"
1,who wrote he ain't heavy he's my brother lyrics,"[Bobby Scott, Bob Russell]"
2,how many seasons of the bastard executioner ar...,"[one, one season]"
3,when did the eagles win last super bowl,[2017]
4,who won last year's ncaa women's basketball,[South Carolina]


## Preprocessing: Answer Normalization / Data Cleaning

In [5]:
def normalize_answer(ans): # a custom function for standardizing answers

    if isinstance(ans, list): # whether the answer is a Python list
        return ans[0] # returns the first answer in the list

    return ans # if already a string return unchanged

df['answer'] = df['answer'].apply(normalize_answer) # apply to all answers


## Improved Wikipedia Retrieval

This function implements the retrieval stage of the RAG pipeline by dynamically fetching relevant Wikipedia documents.

In [ ]:
def retrieve_wikipedia_context(query):

    documents = [] # Store retrieved documents
  
  # Perform Wikipedia search using the user query.This retrieves semantically related page titles.
    try: 

        search_results = wikipedia.search(query, results=5)

        for title in search_results:  # Loop through retrieved Wikipedia page titles

            try: # Retrieve page content for each title

                page = wikipedia.page(
                    title,
                    auto_suggest=False
                )

                documents.append(page.summary)

            except:
                continue
 # Handle Wikipedia search failures
    except:
        pass

  # Fallback when no documents are retrieved
    if len(documents) == 0:

        documents.append(
            f"No information found for {query}"
        )

    return "\n".join(documents)


## Retrieve Context for All Dataset Questions

This block performs bulk retrieval by fetching Wikipedia context for every dataset question and storing it in the dataframe. Precomputing retrieval contexts improves efficiency and supports downstream generation and evaluation stages in the RAG pipeline.

In [ ]:
contexts = []

for question in tqdm(df['question']): # tqdm displays progress bar

    context = retrieve_wikipedia_context(question)

    contexts.append(context)

df['context'] = contexts # Add retrieved contexts as new dataframe column

print("Contexts Retrieved")


100%|██████████| 100/100 [04:30<00:00,  2.70s/it]

Contexts Retrieved


## Text Chunking

This function performs overlapping text chunking by splitting long retrieved documents into smaller semantic segments. Chunking improves embedding quality and retrieval precision by allowing FAISS to retrieve localized context rather than entire documents. Overlapping chunks preserve contextual continuity across chunk boundaries.”

In [ ]:
# This function splits long text into smaller overlapping chunks.
def chunk_text(text, chunk_size=CHUNK_SIZE, overlap=OVERLAP):

    words = text.split()

    chunks = []

    start = 0

    while start < len(words):  # Continue chunking until all words are processed

        end = start + chunk_size  # Define chunk end position

        chunk = words[start:end]  # Extract chunk words

        chunks.append(" ".join(chunk))  # Convert chunk words back into sentence form

        start += chunk_size - overlap  # Move chunk window forward # overlap preserves contextual continuity between neighboring chunks

    return chunks


## Generate Chunks from Retrieved Context

This block preprocesses retrieved Wikipedia documents by splitting them into smaller overlapping semantic chunks. All chunks are aggregated into a master list, which later becomes the input for embedding generation and FAISS vector indexing in the advanced RAG pipeline.

In [ ]:
all_chunks = [] # master list

for idx, row in tqdm(df.iterrows(), total=len(df)): # Loop through every row in the dataframe

    chunks = chunk_text(row['context']) # splits large retrieved document into smaller overlapping chunks

    all_chunks.extend(chunks)

print("Total Chunks:", len(all_chunks))


100%|██████████| 100/100 [00:00<00:00, 1887.90it/s]

Total Chunks: 5031


## Embedding : Generate Semantic Embeddings for Text Chunks

This block generates dense semantic embeddings for all text chunks using the BGE SentenceTransformer model. These embeddings convert textual chunks into numerical vector representations that capture semantic meaning, enabling efficient similarity-based retrieval using FAISS.

In [ ]:
# LOAD EMBEDDING MODEL

embed_model = SentenceTransformer( # This model converts text into dense semantic vectors
    EMBEDDING_MODEL
)

# GENERATE CHUNK EMBEDDINGS
chunk_embeddings = embed_model.encode(
    all_chunks,
    batch_size=8,
    show_progress_bar=True,
    normalize_embeddings=True
)
# Convert embeddings into NumPy float32 format because FAISS requires float32 vectors
chunk_embeddings = np.array(
    chunk_embeddings
).astype('float32')

print(chunk_embeddings.shape)


Batches: 100%|██████████| 629/629 [01:14<00:00,  8.39it/s]

(5031, 384)


## Build FAISS Vector Index for Semantic Retrieval

This block initializes a FAISS(Facebook AI Similarity Search: high-speed vector retrieval library) vector index using the generated chunk embeddings. The embeddings are stored inside the FAISS database to enable efficient semantic similarity retrieval during question answering. Since normalized embeddings are used, the FAISS inner-product index approximates cosine similarity search


In [ ]:
embedding_dim = chunk_embeddings.shape[1]

# IndexFlatIP performs Inner Product similarity search
index = faiss.IndexFlatIP(
    embedding_dim
)

index.add(chunk_embeddings) # Store all chunk embeddings inside FAISS vector database

print("Index Size:", index.ntotal) # Print total number of indexed vectors


Index Size: 5031


Cosine similarity measures semantic similarity between embedding vectors by comparing the angle between them. In the advanced RAG pipeline, cosine similarity enables FAISS to retrieve semantically relevant chunks even when exact keywords differ between the query and the retrieved text.

## Dense Retrieval : Semantic Chunk Retrieval using FAISS

This function performs semantic retrieval by converting the user query into an embedding vector and searching the FAISS index for the most semantically similar chunks. The retrieved chunks are then used as contextual grounding for the generation model.

In [ ]:
def retrieve_chunks(query, top_k=TOP_K): # This function retrieves top semantically relevant chunks from the FAISS vector database
 
 # Convert user query into semantic embedding vector
    query_embedding = embed_model.encode( 
        [query],
        normalize_embeddings=True
    ).astype('float32')

 # Search vector database for top-k most similar chunks
    scores, indices = index.search(
        query_embedding,
        top_k
    )
# STORE RETRIEVED CHUNKS

    retrieved = []

    for idx in indices[0]: # Loop through retrieved indices

        retrieved.append(all_chunks[idx]) # Retrieve original chunk text using retrieved index

    return retrieved


## CrossEncoder Reranking

This block uses a CrossEncoder reranker to improve retrieval precision after FAISS retrieval. The reranker jointly processes the query and retrieved chunks to assign relevance scores, enabling more accurate ranking of contextual information before generation.

In [ ]:
# Load pretrained CrossEncoder model.This model evaluates query-chunk relevance directly

reranker = CrossEncoder(
    'cross-encoder/ms-marco-MiniLM-L-6-v2'
)
# This function reranks retrieved chunks based on relevance to the query

def rerank_chunks(query, chunks, top_n=TOP_N):

    pairs = [[query, chunk] for chunk in chunks] # CrossEncoder requires:[query, chunk] input pairs

    scores = reranker.predict(pairs)  # Predict relevance score for each query-chunk pair

# Combine: chunk + score
# Then sort descending based on relevance score
    ranked = sorted(
        zip(chunks, scores),
        key=lambda x: x[1],
        reverse=True
    )

# Return only: top_n highest-ranked chunks
    return [
        chunk
        for chunk, score in ranked[:top_n]
    ]


Loading weights: 100%|██████████| 105/105 [00:00<00:00, 7223.97it/s]


: 

## Load FLAN-T5 Tokenizer and Generation Model

This block loads the FLAN-T5 tokenizer and sequence-to-sequence generation model from HuggingFace Transformers. The tokenizer converts textual prompts into token IDs understandable by the transformer model , while the FLAN-T5 model generates grounded answers using the retrieved contextual information.

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(
    GENERATION_MODEL
)

model_gen = AutoModelForSeq2SeqLM.from_pretrained(
    GENERATION_MODEL
)


Loading weights: 100%|██████████| 282/282 [00:00<00:00, 3560.51it/s]

## Prompt Construction for Grounded Generation

This function constructs the final grounded prompt by combining retrieved contextual information with the user question and generation instructions. The prompt explicitly restricts the model to use only retrieved context and includes fallback instructions to reduce hallucination.

In [ ]:
def build_prompt(context, question):
# Instruct model to answer ONLY using retrieved context. 
# If answer is not available in retrieved context,instruct model to say:"I don't know."to have hallucination control.
    return f'''
Answer ONLY using the provided context.  

If the answer is not present,
say:
I don't know.

Context:
{context}

Question:
{question}

Short factual answer:
'''

Modern instruction-tuned LLMs such as FLAN-T5 can directly understand natural-language prompts because they are trained on instruction-following datasets. Prompt engineering therefore acts as a mechanism to control model behavior, grounding, factuality, and answer formatting within the RAG pipeline.

## Answer Generation
This function implements the complete Advanced RAG inference pipeline by performing semantic retrieval with FAISS, CrossEncoder reranking, contextual prompt construction, tokenization, and grounded answer generation using FLAN-T5.”

In [ ]:
def generate_answer(question):

# Retrieve top semantically relevant chunks using FAISS
    retrieved_chunks = retrieve_chunks( 

        question,
        top_k=TOP_K
    )
# Improve retrieval precision using CrossEncoder reranking
    reranked_chunks = rerank_chunks(
        question,
        retrieved_chunks,
        top_n=TOP_N
    )
# Combine reranked chunks into one contextual passage
    final_context = "\n\n".join(
        reranked_chunks
    )
 # Construct grounded prompt using retrieved context and user question
    prompt = build_prompt(
        final_context,
        question
    )
# Convert textual prompt into token IDs for FLAN-T5
    inputs = tokenizer(
        prompt,
        return_tensors='pt',
        truncation=True,  # Truncate long prompts to fit model limits
        max_length=1024
    )
# Generate grounded answer using FLAN-T5
    outputs = model_gen.generate(
        **inputs,
        max_new_tokens=32,
        temperature=0.0
    )
# Convert generated token IDs back into human-readable text
    answer = tokenizer.decode(
        outputs[0],
        skip_special_tokens=True
    )
 # Return:
    # 1. generated answer
    # 2. retrieved contextual information
    return answer, final_context


The retrieval pipeline does not directly generate answers. It only provides semantically relevant contextual information. The FLAN-T5 generation model then reads the retrieved context, understands the question, performs semantic reasoning, extracts the relevant information, and generates the final grounded answer in natural language.

## Evaluation

#### Text Cleaning and Normalization for Evaluation

This function normalizes predicted and ground-truth answers before evaluation by converting text to lowercase, removing punctuation, and standardizing whitespace. Text normalization ensures fair and consistent computation of evaluation metrics such as Exact Match and F1 score.

In [ ]:
def clean_text(text):

    text = text.lower() # CONVERT TO LOWERCASE
    
# Keep only: lowercase letters,numbers,spaces
# Replace punctuation/symbols with spaces
    text = re.sub(
        r'[^a-z0-9 ]',
        ' ',
        text
    )

# Replace multiple spaces with single space
    text = re.sub(
        r'\s+',
        ' ',
        text
    ).strip()

    return text


#### Exact Match Evaluation Metric
This function computes the Exact Match metric by comparing normalized predicted and ground-truth answers. It returns 1 when both answers match exactly after text cleaning, otherwise 0.

In [ ]:
def exact_match(prediction, ground_truth):

    pred = clean_text(prediction)

    gt = clean_text(ground_truth)

    return int(pred == gt)


#### F1 Score Evaluation Metric

This function computes token-level F1 score between predicted and ground-truth answers by measuring token overlap. It balances precision and recall, allowing partial credit for semantically correct but non-identical generated answers. ALIAS measures partial correctness.

In [ ]:
def f1_score(prediction, ground_truth):

# Clean and split prediction into tokens
    pred_tokens = clean_text(  
        prediction
    ).split()

# Clean and split actual answer into tokens
    gt_tokens = clean_text(
        ground_truth
    ).split()

# Compute overlapping tokens between prediction and ground truth
    common = set(pred_tokens) & set(gt_tokens)

# If no common tokens exist,F1 score becomes 0
    if len(common) == 0:
        return 0
    
 # Precision measures: how many predicted tokens are correct
    precision = len(common) / len(pred_tokens)

# Recall measures: how much of ground truth was successfully retrieved
    recall = len(common) / len(gt_tokens)

# Harmonic mean of: precision and recall
    return (
        2 * precision * recall
    ) / (precision + recall)


#### Retrieval Hit Rate Evaluation Metric

This function computes Retrieval Hit Rate by checking whether the ground-truth answer appears inside the retrieved context. It evaluates retrieval effectiveness independently from generation quality, helping diagnose whether failures originate from retrieval or answer generation.

In [ ]:
def retrieval_hit(context, answer):

    context = clean_text(context)

    answer = clean_text(answer)

    return int(answer in context)


## Evaluate Pipeline

This block evaluates the complete Advanced RAG pipeline by generating answers for multiple dataset questions and computing Exact Match, F1 Score, and Retrieval Hit Rate. The results are aggregated to measure both retrieval effectiveness and grounded answer quality.

In [ ]:
results = []

EVAL_SAMPLES = 50

for idx in tqdm(range(EVAL_SAMPLES)):

    question = df.iloc[idx]['question']

    answer = df.iloc[idx]['answer']

    prediction, context = generate_answer(
        question
    )

    em = exact_match(
        prediction,
        answer
    )

    f1 = f1_score(
        prediction,
        answer
    )

    hit = retrieval_hit(
        context,
        answer
    )

    results.append({

        'question': question,
        'ground_truth': answer,
        'prediction': prediction,
        'exact_match': em,
        'f1_score': f1,
        'retrieval_hit': hit
    })

results_df = pd.DataFrame(results)

print("Average Exact Match:",
      results_df['exact_match'].mean())

print("Average F1 Score:",
      results_df['f1_score'].mean())

print("Retrieval Hit Rate:",
      results_df['retrieval_hit'].mean())


100%|██████████| 50/50 [02:47<00:00,  3.35s/it]

Average Exact Match: 0.1
Average F1 Score: 0.16229351155666943
Retrieval Hit Rate: 0.34


The Advanced RAG pipeline achieved moderate retrieval and generation performance. Retrieval Hit Rate indicates that approximately 34% of retrieved contexts contained the correct answer, suggesting retrieval quality remains the primary bottleneck. Exact Match scores remained low due to the strict nature of exact-string evaluation, while F1 scores demonstrated partial semantic correctness in generated responses.

🚀 WHY RETRIEVAL HIT IS LOW

Likely causes:

1. Wikipedia Retrieval Weakness : Used wikipedia.search() which is
❌ noisy
❌ inconsistent
❌ not optimized for QA

2. Small Embedding Model :
bge-small-en-v1.5 is lightweight.Better embeddings improve retrieval.

3. Small Generation Model : 
flan-t5-base is decent, but not strong enough for robust QA reasoning.

4. Limited Retrieval Corpus : Wikipedia API retrieval:
❌ unstable
❌ incomplete
❌ inconsistent